Только немое I2V. T4 ×2, Internet On, 48 GiB свободного scratch. Добавьте отдельный v6 inputs.zip как Input. Два коротких теста сохранения лиц; A14B/Q8/FP32 на двух T4 ещё НЕ проверен реальным запуском. Локальные кадры, голос, музыка, титры и монтаж не генерируются здесь. Run All → results.zip; клипы проверяются до монтажа. Текущая GPU-квота 0 до 2026-10-10 00:00 UTC: запуск пока не отправлять.

In [ ]:
EXPECTED_HANDOFF_ID = 'video-494bee349b70131e'


In [ ]:
import json, sys
from pathlib import Path
SCRATCH = Path('/tmp/videogenerator-v6-video-only')
SCRATCH.mkdir(parents=True,exist_ok=True)
BUNDLE = {'contract.py': '"""Portable image -> video -> local-edit contract. No model imports or credentials."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport stat\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\n\ndef digest(path):\n    hasher = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(2**20), b\'\'):\n            hasher.update(block)\n    return hasher.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                     separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef safe_id(value):\n    reserved = {\'con\', \'prn\', \'aux\', \'nul\'} | {f\'com{i}\' for i in range(1, 10)} | {f\'lpt{i}\' for i in range(1, 10)}\n    if not isinstance(value, str) or not re.fullmatch(r\'[a-z0-9][a-z0-9-]{1,63}\', value) or value in reserved:\n        raise ValueError(\'Unsafe or invalid identifier: \' + repr(value))\n    return value\n\n\ndef safe_relative(value):\n    path = PurePosixPath(value)\n    if (not isinstance(value, str) or \'\\\\\' in value or \':\' in value or path.is_absolute()\n            or any(p in (\'\', \'.\', \'..\') for p in value.split(\'/\'))):\n        raise ValueError(\'Unsafe archive path: \' + repr(value))\n    return path\n\n\ndef validate_inputs(manifest):\n    if manifest.get(\'schema\') != \'video-inputs/v1\':\n        raise ValueError(\'Expected video-inputs/v1 manifest.\')\n    expected = \'video-\' + identity({k: v for k, v in manifest.items() if k != \'handoff_id\'})[:16]\n    if manifest.get(\'handoff_id\') != expected:\n        raise ValueError(\'Input identity does not match content.\')\n    if manifest[\'policy\'][\'kaggle_scope\'] != \'silent_image_to_video_only\':\n        raise ValueError(\'Kaggle stage must remain video only.\')\n    assets, requests = manifest[\'assets\'], manifest[\'requests\']\n    if not assets or not requests:\n        raise ValueError(\'Empty generation batch.\')\n    if len({a[\'id\'] for a in assets}) != len(assets) or len({r[\'id\'] for r in requests}) != len(requests):\n        raise ValueError(\'Duplicate asset/request identifiers.\')\n    known = {a[\'id\'] for a in assets}\n    paths = set()\n    for asset in assets:\n        safe_id(asset[\'id\'])\n        path = safe_relative(asset[\'file\'])\n        if path.parts[0] != \'frames\' or path.suffix != \'.png\' or asset[\'file\'] in paths:\n            raise ValueError(\'Expected unique PNG keyframes in frames/.\')\n        paths.add(asset[\'file\'])\n        if not re.fullmatch(\'[0-9a-f]{64}\', asset[\'sha256\']):\n            raise ValueError(\'Invalid source checksum.\')\n    for request in requests:\n        safe_id(request[\'id\'])\n        if request[\'asset_id\'] not in known or not request[\'prompt\'].strip():\n            raise ValueError(\'Missing source asset or motion prompt.\')\n        c = request[\'config\']\n        if c[\'WIDTH\'] % 16 or c[\'HEIGHT\'] % 16 or c[\'WIDTH\'] * 9 != c[\'HEIGHT\'] * 16:\n            raise ValueError(\'Use native 16:9 dimensions divisible by 16.\')\n        if not 256 <= c[\'WIDTH\'] <= 1280 or c[\'FRAMES\'] < 9 or (c[\'FRAMES\'] - 1) % 4:\n            raise ValueError(\'Invalid native size or frame count.\')\n        if c[\'FPS\'] != 24 or not 20 <= c[\'STEPS\'] <= 80 or not 1 < c[\'GUIDANCE\'] <= 10:\n            raise ValueError(\'Invalid generation settings.\')\n        if not isinstance(c[\'SEED\'], int) or c[\'SEED\'] < 0:\n            raise ValueError(\'Use a nonnegative integer seed.\')\n    return manifest\n\n\ndef check_zip(bundle):\n    names = set()\n    total = 0\n    for info in bundle.infolist():\n        safe_relative(info.filename.rstrip(\'/\') if info.is_dir() else info.filename)\n        if info.filename in names or stat.S_ISLNK(info.external_attr >> 16):\n            raise ValueError(\'Duplicate archive members or symbolic links are forbidden.\')\n        names.add(info.filename)\n        total += info.file_size\n    if total > 8 * 2**30 or len(names) > 5000:\n        raise ValueError(\'Archive exceeds the local handoff safety limits.\')\n\n\ndef read_json_member(bundle, name):\n    if bundle.getinfo(name).file_size > 4 * 2**20:\n        raise ValueError(\'Oversized JSON metadata.\')\n    return json.loads(bundle.read(name))\n\n\ndef load_inputs(input_root, expected_id, destination):\n    """Accept either Kaggle-extracted files or an attached ZIP; bind all frames to SHA256."""\n    input_root, destination = Path(input_root), Path(destination)\n    validate = lambda m: validate_inputs(m) if m.get(\'handoff_id\') == expected_id else None\n    for path in input_root.rglob(\'video-inputs.json\'):\n        if path.stat().st_size > 4 * 2**20:\n            continue\n        manifest = validate(json.loads(path.read_text(encoding=\'utf-8\')))\n        if manifest is None:\n            continue\n        for asset in manifest[\'assets\']:\n            source = path.parent / asset[\'file\']\n            if not source.is_file() or digest(source) != asset[\'sha256\']:\n                raise ValueError(\'Missing/changed keyframe: \' + asset[\'id\'])\n        for asset in manifest[\'assets\']:\n            target = destination / asset[\'file\']\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if not target.exists() or digest(target) != asset[\'sha256\']:\n                shutil.copy2(path.parent / asset[\'file\'], target)\n        save_json(destination / \'video-inputs.json\', manifest)\n        return manifest\n    for path in input_root.rglob(\'*inputs.zip\'):\n        with zipfile.ZipFile(path) as bundle:\n            check_zip(bundle)\n            if \'video-inputs.json\' not in bundle.namelist():\n                continue\n            manifest = validate(read_json_member(bundle, \'video-inputs.json\'))\n            if manifest is None:\n                continue\n            frames = []\n            for asset in manifest[\'assets\']:\n                if bundle.getinfo(asset[\'file\']).file_size > 64 * 2**20:\n                    raise ValueError(\'Oversized image asset.\')\n                data = bundle.read(asset[\'file\'])\n                if hashlib.sha256(data).hexdigest() != asset[\'sha256\']:\n                    raise ValueError(\'Source keyframe checksum mismatch.\')\n                frames.append((asset[\'file\'], data))\n            for filename, data in frames:\n                target = destination / filename\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(data)\n            save_json(destination / \'video-inputs.json\', manifest)\n            return manifest\n    raise FileNotFoundError(\'Attach the prepared video-inputs bundle: \' + expected_id)\n\n\ndef validate_return_manifest(manifest, inputs, fingerprint=None):\n    validate_inputs(inputs)\n    if (manifest.get(\'schema\') != \'video-results/v1\' or\n            manifest.get(\'handoff_id\') != inputs[\'handoff_id\']):\n        raise ValueError(\'Returned clips do not belong to these inputs.\')\n    if fingerprint is not None and manifest.get(\'job_fingerprint\') != fingerprint:\n        raise ValueError(\'Worker/model fingerprint changed; old results cannot be resumed.\')\n    known = {r[\'id\']: r for r in inputs[\'requests\']}\n    assets = {a[\'id\']: a for a in inputs[\'assets\']}\n    ids = set()\n    for clip in manifest[\'clips\']:\n        ident = safe_id(clip[\'id\'])\n        if ident in ids or ident not in known:\n            raise ValueError(\'Unknown/duplicate returned clip.\')\n        ids.add(ident)\n        request = known[ident]\n        if (clip[\'asset_id\'] != request[\'asset_id\'] or clip[\'config\'] != request[\'config\']\n                or clip[\'source_sha256\'] != assets[request[\'asset_id\']][\'sha256\']):\n            raise ValueError(\'Returned clip source/settings differ from the request.\')\n        if clip[\'file\'] != f\'clips/{ident}.mp4\' or clip[\'report\'] != f\'reports/{ident}.json\':\n            raise ValueError(\'Unexpected returned media path.\')\n        if not clip[\'technical_pass\'] or not re.fullmatch(\'[0-9a-f]{64}\', clip[\'sha256\']):\n            raise ValueError(\'Returned clip did not pass technical checks.\')\n    return manifest\n\n\ndef make_archive(run, target):\n    run = Path(run)\n    files = {p.relative_to(run).as_posix(): digest(p) for p in sorted(run.rglob(\'*\'))\n        if p.is_file() and \'__pycache__\' not in p.parts and \'.partial\' not in p.name\n        and p.name != \'checksums.json\' and not any(part in (\'embeddings\', \'frames\', \'source\') for part in p.relative_to(run).parts)}\n    save_json(run / \'checksums.json\', files)\n    with zipfile.ZipFile(target, \'w\', zipfile.ZIP_DEFLATED, compresslevel=3) as bundle:\n        for filename in files:\n            bundle.write(run / filename, filename)\n        bundle.write(run / \'checksums.json\', \'checksums.json\')\n    with zipfile.ZipFile(target) as bundle:\n        if bundle.testzip():\n            raise ValueError(\'Result archive CRC check failed.\')\n    return Path(target)\n', 'driver.py': '"""Run one bounded batch; retain completed clips and diagnostics on failure."""\nimport json\nimport os\nimport shutil\nimport subprocess\nimport threading\nimport zipfile\nfrom pathlib import Path\n\nfrom contract import (check_zip, digest, make_archive, read_json_member, save_json,\n                      validate_return_manifest)\n\n\ndef clip_records(job):\n    run = Path(job[\'run\'])\n    inputs = job[\'inputs\']\n    records = []\n    for request in inputs[\'requests\']:\n        ident = request[\'id\']\n        video, sidecar = run / \'clips\' / f\'{ident}.mp4\', run / \'reports\' / f\'{ident}.json\'\n        if not video.exists() or not sidecar.exists():\n            continue\n        report = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n        if report.get(\'job_fingerprint\') != job[\'fingerprint\'] or report[\'sha256\'] != digest(video):\n            continue\n        records.append({\'id\': ident, \'asset_id\': request[\'asset_id\'], \'config\': request[\'config\'],\n            \'file\': f\'clips/{ident}.mp4\', \'report\': f\'reports/{ident}.json\',\n            \'sha256\': report[\'sha256\'], \'source_sha256\': report[\'source_sha256\'],\n            \'technical_pass\': report[\'technical_pass\'], \'production_approved\': False})\n    return records\n\n\ndef restore_completed(job, input_root):\n    """Optional attach of the SAME batch\'s previous result ZIP. Never executes imported code."""\n    run = Path(job[\'run\'])\n    for archive in Path(input_root).rglob(\'*results.zip\'):\n        with zipfile.ZipFile(archive) as bundle:\n            check_zip(bundle)\n            if \'video-results.json\' not in bundle.namelist():\n                continue\n            manifest = read_json_member(bundle, \'video-results.json\')\n            if manifest.get(\'job_fingerprint\') != job[\'fingerprint\']:\n                continue\n            validate_return_manifest(manifest, job[\'inputs\'], job[\'fingerprint\'])\n            for clip in manifest[\'clips\']:\n                report = read_json_member(bundle, clip[\'report\'])\n                import hashlib\n                data = bundle.read(clip[\'file\'])\n                if (hashlib.sha256(data).hexdigest() != clip[\'sha256\'] or\n                        report.get(\'job_fingerprint\') != job[\'fingerprint\'] or\n                        report.get(\'sha256\') != clip[\'sha256\'] or\n                        report.get(\'source_sha256\') != clip[\'source_sha256\'] or\n                        report.get(\'config\') != clip[\'config\'] or not report.get(\'technical_pass\')):\n                    raise ValueError(\'Invalid previous result; will not resume.\')\n                (run / clip[\'file\']).write_bytes(data)\n                save_json(run / clip[\'report\'], report)\n                print(\'RESTORED\', clip[\'id\'], flush=True)\n    # An interrupted write with no matching report must not count as cached output.\n    good = {r[\'file\'] for r in clip_records(job)}\n    for video in (run / \'clips\').glob(\'*.mp4\'):\n        if video.relative_to(run).as_posix() not in good:\n            # Keep it as diagnostics rather than deleting a recoverable candidate.\n            destination = run / \'diagnostics\' / (video.stem + \'-unverified.mp4\')\n            if destination.exists():\n                raise RuntimeError(\'Unverified diagnostic already exists; inspect it before rerunning.\')\n            shutil.move(str(video), str(destination))\n\n\ndef execute(job, python):\n    run = Path(job[\'run\'])\n    restore_completed(job, \'/kaggle/input\')\n    failure = None\n    environment = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0,1\', HF_HOME=job[\'model_cache\'],\n        HF_HUB_DISABLE_XET=\'1\', TOKENIZERS_PARALLELISM=\'false\', PYTHONUNBUFFERED=\'1\',\n        PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n    expired = threading.Event()\n    with (run / \'logs/worker.log\').open(\'w\', encoding=\'utf-8\') as stream:\n        process = subprocess.Popen([python, \'-u\', str(run / \'scripts/batch_worker.py\'), str(run / \'job.json\')],\n            env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n        def expire():\n            expired.set()\n            if process.poll() is None:\n                process.kill()\n        timer = threading.Timer(job[\'inputs\'][\'policy\'][\'maximum_batch_worker_seconds\'], expire)\n        timer.daemon = True\n        timer.start()\n        try:\n            for line in process.stdout:\n                stream.write(line)\n                stream.flush()\n                if any(word in line for word in (\'DOWNLOAD\', \'EMBEDDING\', \'LOAD \', \'STEP \', \'DONE \',\n                                                  \'RETURN \', \'CACHED \', \'Traceback\', \'Error:\')):\n                    print(line.rstrip(), flush=True)\n            if process.wait() or expired.is_set():\n                failure = \'batch timeout\' if expired.is_set() else \'See logs/worker.log\'\n        except BaseException:\n            if process.poll() is None:\n                process.kill()\n            process.wait()\n            raise\n        finally:\n            timer.cancel()\n            process.stdout.close()\n    records = clip_records(job)\n    result = {\'schema\': \'video-results/v1\', \'handoff_id\': job[\'inputs\'][\'handoff_id\'],\n        \'job_fingerprint\': job[\'fingerprint\'], \'clips\': records,\n        \'requested_clips\': len(job[\'inputs\'][\'requests\']), \'completed_clips\': len(records),\n        \'failure\': failure, \'production_approved\': False, \'checkpoint_modified\': False,\n        \'kaggle_scope\': \'silent_image_to_video_only\', \'paid_apis\': False,\n        \'limits\': [\'Experimental Q8/FP32 backend has not yet passed Kaggle T4 x2 runtime validation.\',\n                   \'Technical validation is not visual approval. No automatic edit insertion.\',\n                   \'Only finished, checksum-bound clips can resume; mid-clip latents are not resumed.\']}\n    validate_return_manifest(result, job[\'inputs\'], job[\'fingerprint\'])\n    save_json(run / \'video-results.json\', result)\n    archive = make_archive(run, run.parent / (run.name + \'-results.zip\'))\n    print(\'EXPORT\', archive.name, len(records), \'raw clip(s)\', flush=True)\n    if failure:\n        raise RuntimeError(\'Batch failed; completed clips and diagnostics saved: \' + str(archive))\n    return archive\n', 'batch_worker.py': '"""Kaggle stage: condition images/prompts, generate silent clips, export diagnostics."""\nimport json\nimport shutil\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nimport wan_worker as engine\nfrom contract import digest, save_json, validate_inputs\n\n\ndef main():\n    job = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    run = Path(job[\'run\'])\n    manifest = validate_inputs(job[\'inputs\'])\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError(\'Two T4 GPUs required; no automatic single-GPU/other-model fallback.\')\n    engine.install_chunked_attention(job[\'config\'][\'ATTENTION_QUERY_CHUNK\'])\n    with engine.Telemetry(run):\n        pending = [r for r in manifest[\'requests\'] if not (run / \'clips\' / f\'{r["id"]}.mp4\').is_file()]\n        if not pending:\n            print(\'CACHED all requested clips\', flush=True)\n            return\n        snapshot, experts = engine.download_models(job, run)\n        assets = {a[\'id\']: a for a in manifest[\'assets\']}\n        shots = [{\'id\': r[\'id\'], \'sha256\': assets[r[\'asset_id\']][\'sha256\'], \'prompt\': r[\'prompt\']}\n                 for r in pending]\n        embedding_job = dict(job, shots=shots)\n        engine.prepare_embeddings(embedding_job, run, snapshot)\n        for number, request in enumerate(pending):\n            ident = request[\'id\']\n            asset = assets[request[\'asset_id\']]\n            if digest(run / asset[\'file\']) != asset[\'sha256\']:\n                raise ValueError(\'Source frame changed after input validation.\')\n            clip_run = run / \'requests\' / ident\n            for folder in (\'source\', \'embeddings\', \'clips\', \'diagnostics\', \'review\', \'logs\'):\n                (clip_run / folder).mkdir(parents=True, exist_ok=True)\n            shutil.copy2(run / asset[\'file\'], clip_run / \'source\' / f\'{ident}.png\')\n            for name in (ident, \'negative\'):\n                shutil.copy2(run / \'embeddings\' / f\'{name}.pt\', clip_run / \'embeddings\' / f\'{name}.pt\')\n            shot = {\'id\': ident, \'sha256\': asset[\'sha256\'], \'prompt\': request[\'prompt\']}\n            clip_job = dict(job, run=str(clip_run), shots=[shot], config=request[\'config\'])\n            if number == 0:\n                sanity = dict(request[\'config\'], WIDTH=256, HEIGHT=144, FRAMES=9, STEPS=2)\n                engine.generate(clip_job, clip_run, snapshot, experts, \'sanity\', sanity)\n                engine.clean_cuda()\n            report = engine.generate(clip_job, clip_run, snapshot, experts, \'quality\', request[\'config\'])\n            generated = clip_run / \'clips\' / f\'{ident}-quality-seed-{request["config"]["SEED"]}.mp4\'\n            target = run / \'clips\' / f\'{ident}.mp4\'\n            generated.replace(target)\n            report.update(id=ident, asset_id=asset[\'id\'], handoff_id=manifest[\'handoff_id\'],\n                job_fingerprint=job[\'fingerprint\'], prompt=request[\'prompt\'],\n                negative_prompt=manifest[\'negative_prompt\'], production_approved=False)\n            save_json(run / \'reports\' / f\'{ident}.json\', report)\n            engine.clean_cuda()\n            print(\'RETURN\', ident, flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'wan_worker.py': '"""Wan A14B Q8/FP32. One active expert at a time, its blocks split across two T4s.\n\nDenoising, conditioning and decoding match Diffusers 0.40 Wan I2V. The scheduler\nis never restarted at the high/low-noise boundary. No TeaCache or distilled LoRA.\n"""\nimport contextlib\nimport gc\nimport json\nimport os\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom common import atomic_json, encode_frames, inspect_video, sha256\n\n\ndef stage_for(timestep, boundary):\n    return \'high\' if float(timestep) >= boundary else \'low\'\n\n\ndef device_map_for(num_layers=40):\n    result = {\'\': 0, \'norm_out\': 1, \'proj_out\': 1}\n    result.update({f\'blocks.{i}\': 0 if i < num_layers // 2 else 1 for i in range(num_layers)})\n    return result\n\n\ndef install_chunked_attention(chunk_size):\n    original = torch.nn.functional.scaled_dot_product_attention\n\n    def chunked(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):\n        if is_causal or dropout_p or query.shape[-2] <= chunk_size:\n            return original(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p,\n                            is_causal=is_causal, **kwargs)\n        parts = []\n        for start in range(0, query.shape[-2], chunk_size):\n            mask = attn_mask\n            if mask is not None and mask.ndim >= 2 and mask.shape[-2] == query.shape[-2]:\n                mask = mask[..., start:start + chunk_size, :]\n            parts.append(original(query[..., start:start + chunk_size, :], key, value,\n                attn_mask=mask, dropout_p=0.0, is_causal=False, **kwargs))\n        return torch.cat(parts, dim=-2)\n\n    torch.nn.functional.scaled_dot_product_attention = chunked\n\n\ndef clean_cuda():\n    gc.collect()\n    for i in range(2):\n        with torch.cuda.device(i):\n            torch.cuda.empty_cache()\n    # Return freed Python/C allocations where the Linux allocator supports it.\n    with contextlib.suppress(Exception):\n        import ctypes\n        ctypes.CDLL(None).malloc_trim(0)\n\n\nclass Telemetry:\n    def __init__(self, run):\n        self.run = run\n        self.stop = threading.Event()\n        self.thread = threading.Thread(target=self.sample, daemon=True)\n\n    def sample(self):\n        with (self.run / \'gpu-telemetry.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n            while not self.stop.is_set():\n                try:\n                    raw = subprocess.check_output([\'nvidia-smi\',\n                        \'--query-gpu=index,utilization.gpu,memory.used,memory.total\',\n                        \'--format=csv,noheader,nounits\'], text=True, timeout=10)\n                    values = [[int(x.strip()) for x in line.split(\',\')] for line in raw.splitlines()]\n                    stream.write(json.dumps({\'time\': time.time(), \'gpu\': values}) + \'\\n\')\n                    stream.flush()\n                except Exception as exc:\n                    stream.write(json.dumps({\'telemetry_error\': type(exc).__name__}) + \'\\n\')\n                    stream.flush()\n                self.stop.wait(5)\n\n    def __enter__(self):\n        self.thread.start()\n        return self\n\n    def __exit__(self, *args):\n        self.stop.set()\n        self.thread.join(timeout=15)\n\n\ndef download_models(job, run):\n    from huggingface_hub import hf_hub_download, snapshot_download\n    original = job[\'models\'][\'original\']\n    # Explicit allowlist: never download the ~106 GiB FP32 diffusion experts.\n    snapshot = snapshot_download(original[\'repo\'], revision=original[\'revision\'],\n        allow_patterns=[\'model_index.json\', \'scheduler/*\', \'tokenizer/*\', \'text_encoder/*\',\n                        \'transformer/config.json\', \'transformer_2/config.json\', \'vae/*\'],\n        ignore_patterns=[\'*.md\', \'*.bin\', \'*.jpg\', \'*.png\', \'*.onnx\'],\n        cache_dir=job[\'model_cache\'], max_workers=4)\n    quant = job[\'models\'][\'quantized\']\n    files = {}\n    for stage, record in quant[\'files\'].items():\n        print(\'DOWNLOAD\', stage, \'Q8 expert\', flush=True)\n        filename = hf_hub_download(quant[\'repo\'], record[\'name\'], revision=quant[\'revision\'],\n                                   cache_dir=job[\'model_cache\'])\n        if sha256(filename) != record[\'sha256\']:\n            raise ValueError(\'Q8 expert checksum mismatch: \' + stage)\n        files[stage] = filename\n    atomic_json(run / \'model-files.json\', {\'snapshot\': snapshot, \'experts\': files,\n                                         \'expert_checksums_verified\': True})\n    return snapshot, files\n\n\ndef prepare_embeddings(job, run, snapshot):\n    from transformers import BitsAndBytesConfig, T5TokenizerFast, UMT5EncoderModel\n    from diffusers.pipelines.wan.pipeline_wan import prompt_clean\n    tokenizer = T5TokenizerFast.from_pretrained(snapshot, subfolder=\'tokenizer\')\n    encoder = UMT5EncoderModel.from_pretrained(snapshot, subfolder=\'text_encoder\',\n        dtype=torch.float32, device_map={\'\': 0}, attn_implementation=\'eager\',\n        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)).eval()\n    prompts = {shot[\'id\']: shot[\'prompt\'] for shot in job[\'shots\']}\n    prompts[\'negative\'] = job[\'negative_prompt\']\n    for ident, text in prompts.items():\n        tokens = tokenizer(prompt_clean(text), padding=\'max_length\',\n            max_length=job[\'config\'][\'MAX_SEQUENCE\'], truncation=True,\n            add_special_tokens=True, return_attention_mask=True, return_tensors=\'pt\').to(\'cuda:0\')\n        length = int(tokens.attention_mask.sum().item())\n        with torch.inference_mode():\n            embedding = encoder(tokens.input_ids, tokens.attention_mask).last_hidden_state.float()\n            embedding[:, length:] = 0\n        if not torch.isfinite(embedding).all().item():\n            raise ValueError(\'Non-finite text embedding.\')\n        torch.save(embedding.cpu(), run / \'embeddings\' / f\'{ident}.pt\')\n        print(\'EMBEDDING\', ident, length, flush=True)\n    del encoder, tokenizer, tokens, embedding\n    clean_cuda()\n\n\ndef load_expert(stage, filename, snapshot, run, trial):\n    from accelerate import dispatch_model\n    from diffusers import GGUFQuantizationConfig, WanTransformer3DModel\n    clean_cuda()\n    print(\'LOAD\', trial, stage, flush=True)\n    started = time.monotonic()\n    # Single-file loader first materializes compressed Q8 on CPU. Explicit dispatch\n    # afterwards avoids its version-specific automatic device-map estimation.\n    model = WanTransformer3DModel.from_single_file(filename,\n        config=snapshot, subfolder=\'transformer\' if stage == \'high\' else \'transformer_2\',\n        torch_dtype=torch.float32, quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float32),\n        low_cpu_mem_usage=True, local_files_only=True)\n    if any(p.device.type == \'meta\' for p in model.parameters()):\n        raise RuntimeError(\'Missing model parameters: cannot dispatch a partial checkpoint.\')\n    mapping = device_map_for(model.config.num_layers)\n    model = dispatch_model(model, device_map=mapping, main_device=0,\n                           force_hooks=True, offload_buffers=False)\n    model.set_attention_backend(\'native\')\n    model.eval()\n    residency = {}\n    for name, param in model.named_parameters():\n        key = str(param.device)\n        residency[key] = residency.get(key, 0) + param.numel() * param.element_size()\n    if set(residency) != {\'cuda:0\', \'cuda:1\'}:\n        raise RuntimeError(\'Expert did not distribute onto both GPUs: \' + repr(residency))\n    hits = {0: 0, 1: 0}\n\n    def note_forward(module, args, result):\n        tensor = result[0] if isinstance(result, tuple) else result\n        if tensor.device.type != \'cuda\':\n            raise RuntimeError(\'Transformer block executed outside CUDA.\')\n        hits[tensor.device.index] += 1\n\n    for block in model.blocks:\n        block.register_forward_hook(note_forward)\n    info = {\'stage\': stage, \'trial\': trial, \'load_seconds\': time.monotonic() - started,\n        \'device_map\': mapping, \'parameter_bytes_by_device\': residency,\n        \'weight_format\': \'GGUF Q8_0\', \'compute_dtype\': \'float32\',\n        \'quantization_warning\': \'Q8 is not an unquantized reference; T5 uses NF4/FP32.\'}\n    atomic_json(run / \'diagnostics\' / f\'{trial}-{stage}-distribution.json\', info)\n    print(\'RESIDENT\', stage, {k: round(v / 2**30, 3) for k, v in residency.items()}, flush=True)\n    return model, hits\n\n\ndef denoise(scheduler, latents, condition, positive, negative, steps, guidance,\n            boundary_ratio, load_model, release_model, on_step):\n    """Exact reference CFG/UniPC semantics; expert swap does not reset scheduler."""\n    scheduler.set_timesteps(steps, device=latents.device)\n    boundary = boundary_ratio * scheduler.config.num_train_timesteps\n    model = None\n    active = None\n    try:\n        for index, timestep in enumerate(scheduler.timesteps):\n            stage = stage_for(timestep, boundary)\n            if stage != active:\n                if model is not None:\n                    release_model(model, active)\n                    model = None\n                model = load_model(stage)\n                active = stage\n            inputs = torch.cat([latents, condition], dim=1)\n            timestamp = timestep.expand(latents.shape[0])\n            with model.cache_context(\'cond\'):\n                conditional = model(hidden_states=inputs, timestep=timestamp,\n                    encoder_hidden_states=positive, return_dict=False)[0]\n            with model.cache_context(\'uncond\'):\n                unconditional = model(hidden_states=inputs, timestep=timestamp,\n                    encoder_hidden_states=negative, return_dict=False)[0]\n            prediction = unconditional + guidance * (conditional - unconditional)\n            if not torch.isfinite(prediction).all().item():\n                raise ValueError(f\'Non-finite prediction at {stage} step {index + 1}.\')\n            latents = scheduler.step(prediction, timestep, latents, return_dict=False)[0]\n            if not torch.isfinite(latents).all().item():\n                raise ValueError(f\'Non-finite latents at step {index + 1}.\')\n            del conditional, unconditional, prediction, inputs\n            on_step(index, timestep, stage, latents)\n    finally:\n        if model is not None:\n            release_model(model, active)\n    return latents\n\n\ndef generate(job, run, snapshot, experts, trial, config):\n    from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanImageToVideoPipeline\n    shot = job[\'shots\'][0]\n    image = Image.open(run / \'source\' / f\'{shot["id"]}.png\').convert(\'RGB\')\n    positive = torch.load(run / \'embeddings\' / f\'{shot["id"]}.pt\', weights_only=True).to(\'cuda:0\')\n    negative = torch.load(run / \'embeddings/negative.pt\', weights_only=True).to(\'cuda:0\')\n    vae = AutoencoderKLWan.from_pretrained(snapshot, subfolder=\'vae\', torch_dtype=torch.float32).to(\'cuda:0\')\n    vae.enable_tiling()\n    vae.enable_slicing()\n    scheduler = UniPCMultistepScheduler.from_pretrained(snapshot, subfolder=\'scheduler\')\n    index = json.loads((Path(snapshot) / \'model_index.json\').read_text())\n    pipe = WanImageToVideoPipeline(tokenizer=None, text_encoder=None,\n        transformer=None, transformer_2=None, vae=vae, scheduler=scheduler,\n        boundary_ratio=index[\'boundary_ratio\'], expand_timesteps=False)\n    resized = pipe.video_processor.preprocess(image, height=config[\'HEIGHT\'], width=config[\'WIDTH\']).to(\'cuda:0\', torch.float32)\n    with torch.inference_mode():\n        latents, condition = pipe.prepare_latents(resized, 1, vae.config.z_dim,\n            config[\'HEIGHT\'], config[\'WIDTH\'], config[\'FRAMES\'], torch.float32,\n            torch.device(\'cuda:0\'), torch.Generator(device=\'cuda:0\').manual_seed(config[\'SEED\']))\n    if not torch.isfinite(condition).all().item():\n        raise ValueError(\'Non-finite VAE image conditioning.\')\n    vae.to(\'cpu\')\n    del resized\n    clean_cuda()\n    for gpu in range(2):\n        torch.cuda.reset_peak_memory_stats(gpu)\n    started = time.monotonic()\n    stats = {\'trial\': trial, \'steps\': [], \'expert_forward_hits\': {}, \'completed\': False}\n    current_hits = {}\n\n    def load(stage):\n        model, hits = load_expert(stage, experts[stage], snapshot, run, trial)\n        current_hits[stage] = hits\n        return model\n\n    def release(model, stage):\n        stats[\'expert_forward_hits\'][stage] = dict(current_hits[stage])\n        # Move the compressed weights off GPUs, then remove the only model reference\n        # in denoise before the next load. CPU temporary allocation is ~14.35 GiB.\n        from accelerate.hooks import remove_hook_from_module\n        remove_hook_from_module(model, recurse=True)\n        model.to(\'cpu\')\n\n    def step(index, timestep, stage, values):\n        clean_cuda() if index == 0 else None\n        progress = {\'step\': index + 1, \'timestep\': float(timestep), \'expert\': stage,\n                    \'elapsed_seconds\': time.monotonic() - started}\n        stats[\'steps\'].append(progress)\n        atomic_json(run / \'diagnostics\' / f\'{trial}-progress.json\', stats)\n        print(\'STEP\', trial, index + 1, \'/\', config[\'STEPS\'], stage,\n              round(progress[\'elapsed_seconds\'], 1), flush=True)\n\n    with torch.inference_mode():\n        latents = denoise(scheduler, latents, condition, positive, negative,\n            config[\'STEPS\'], config[\'GUIDANCE\'], index[\'boundary_ratio\'], load, release, step)\n    clean_cuda()\n    if set(stats[\'expert_forward_hits\']) != {\'high\', \'low\'} or any(\n            not hits.get(gpu) for hits in stats[\'expert_forward_hits\'].values() for gpu in range(2)):\n        raise RuntimeError(\'Both experts must execute blocks on both GPUs.\')\n    stats[\'denoise_seconds\'] = time.monotonic() - started\n    stats[\'peak_allocated_gib\'] = [torch.cuda.max_memory_allocated(i) / 2**30 for i in range(2)]\n    vae.to(\'cuda:0\')\n    with torch.inference_mode():\n        mean = torch.tensor(vae.config.latents_mean, device=\'cuda:0\').view(1, vae.config.z_dim, 1, 1, 1)\n        std = torch.tensor(vae.config.latents_std, device=\'cuda:0\').view(1, vae.config.z_dim, 1, 1, 1)\n        decoded = vae.decode(latents * std + mean, return_dict=False)[0]\n        if not torch.isfinite(decoded).all().item():\n            raise ValueError(\'Non-finite VAE decoded frames.\')\n        frames = pipe.video_processor.postprocess_video(decoded, output_type=\'np\')[0]\n    if not np.isfinite(frames).all():\n        raise ValueError(\'Non-finite output pixels.\')\n    folder = \'diagnostics\' if trial == \'sanity\' else \'clips\'\n    target = run / folder / f\'{shot["id"]}-{trial}-seed-{config["SEED"]}.mp4\'\n    encode_frames(np.clip(frames * 255, 0, 255).round().astype(np.uint8), target, 24)\n    report, samples = inspect_video(target, config, image)\n    if report.get(\'warning_black_frames\'):\n        raise RuntimeError(\'Black-frame warning; output rejected.\')\n    for sample_index, sample in enumerate(samples):\n        Image.fromarray(sample).save(run / \'review\' / f\'{trial}-{sample_index}.png\')\n    stats.update(completed=True, total_seconds=time.monotonic() - started)\n    report.update(kind=\'sanity_not_quality\' if trial == \'sanity\' else \'i2v_quality_candidate\',\n        config=config, runtime=stats, source_sha256=shot[\'sha256\'], models=job[\'models\'],\n        identity_preservation_verified=False, production_approved=False,\n        limitations=[\'Two GPUs shard each active expert; compute is sequential between layer groups.\',\n                     \'Q8 diffusion and NF4 T5 are quantized, not full-precision weights.\',\n                     \'No interpolation, upscale, sound, postprocessing, cache or motion LoRA.\'])\n    atomic_json(target.with_suffix(\'.json\'), report)\n    atomic_json(run / \'diagnostics\' / f\'{trial}-progress.json\', stats)\n    print(\'DONE\', target.name, stats[\'total_seconds\'], flush=True)\n    return report\n\n\ndef main():\n    job = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    run = Path(job[\'run\'])\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError(\'Two visible GPUs are mandatory; no silent single-GPU fallback.\')\n    install_chunked_attention(job[\'config\'][\'ATTENTION_QUERY_CHUNK\'])\n    with Telemetry(run):\n        snapshot, experts = download_models(job, run)\n        prepare_embeddings(job, run, snapshot)\n        sanity = dict(job[\'config\'], WIDTH=256, HEIGHT=144, FRAMES=9, STEPS=2)\n        generate(job, run, snapshot, experts, \'sanity\', sanity)\n        clean_cuda()\n        generate(job, run, snapshot, experts, \'quality\', job[\'config\'])\n\n\nif __name__ == \'__main__\':\n    main()\n', 'bootstrap.py': '"""Two-T4, pip-less bootstrap; does not replace Kaggle\'s Torch/CUDA/NumPy."""\nimport importlib.metadata as metadata\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport venv\n\nfrom common import atomic_json, fingerprint\n\n\ndef bootstrap(run, scratch, packages):\n    import torch\n    if platform.system() != \'Linux\' or torch.cuda.device_count() != 2:\n        raise RuntimeError(\'Select Kaggle T4 x2 and Internet On. Exactly two GPUs are required.\')\n    if any(torch.cuda.get_device_capability(i) != (7, 5) for i in range(2)):\n        raise RuntimeError(\'This controlled experiment is configured for two Tesla T4 cards.\')\n    if not all(shutil.which(tool) for tool in (\'ffmpeg\', \'ffprobe\', \'nvidia-smi\')):\n        raise RuntimeError(\'ffmpeg, ffprobe and nvidia-smi are required.\')\n    free = shutil.disk_usage(scratch).free\n    if free < 48 * 2**30:\n        raise RuntimeError(f\'Need 48 GiB free scratch; found {free / 2**30:.1f}. Full FP32 experts are NOT downloaded.\')\n    protected = {}\n    for name in (\'torch\', \'torchvision\', \'torchaudio\', \'numpy\', \'Pillow\'):\n        try:\n            protected[name] = metadata.version(name)\n        except metadata.PackageNotFoundError:\n            pass\n    signature = {\'python\': sys.version, \'packages\': packages, \'protected\': protected,\n                 \'bootstrap\': \'pipless-parent-pip-v1\'}\n    environment = scratch / \'venvs\' / fingerprint(signature)[:16]\n    python = environment / \'bin/python\'\n    marker = environment / \'installed.json\'\n    if not marker.exists() or json.loads(marker.read_text()) != signature:\n        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)\n        prefixes = json.loads(subprocess.check_output([str(python), \'-c\',\n            \'import sys,json;print(json.dumps([sys.prefix,sys.base_prefix]))\'], text=True))\n        if prefixes[0] == prefixes[1] or environment.resolve() != __import__(\'pathlib\').Path(prefixes[0]).resolve():\n            raise RuntimeError(\'Environment isolation failed; installation cancelled.\')\n        constraints = environment / \'protected.txt\'\n        constraints.write_text(\'\\n\'.join(f\'{k}=={v}\' for k, v in protected.items()) + \'\\n\')\n        with (run / \'logs/install.log\').open(\'w\') as stream:\n            result = subprocess.run([sys.executable, \'-m\', \'pip\', \'--python\', str(python),\n                \'install\', \'--disable-pip-version-check\', \'--prefer-binary\', \'-c\', str(constraints),\n                *packages], stdout=stream, stderr=subprocess.STDOUT, timeout=1800)\n        if result.returncode:\n            raise RuntimeError(\'Isolated install failed: \' + (run / \'logs/install.log\').read_text()[-3000:])\n        atomic_json(marker, signature)\n    if any(metadata.version(k) != v for k, v in protected.items()):\n        raise RuntimeError(\'Protected parent package changed.\')\n    probe = subprocess.run([str(python), \'-c\',\n        \'import torch,gguf,bitsandbytes as b; \'\n        \'from diffusers import GGUFQuantizationConfig,WanTransformer3DModel,WanImageToVideoPipeline; \'\n        \'from accelerate import dispatch_model; \'\n        \'assert torch.cuda.device_count()==2; \'\n        \'x=torch.randn(64,64,device="cuda:0"); q,s=b.functional.quantize_4bit(x,quant_type="nf4"); \'\n        \'assert torch.isfinite(b.functional.dequantize_4bit(q,s)).all().item(); \'\n        \'print("Two-GPU imports and text-encoder kernel probe passed")\'],\n        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=180)\n    (run / \'logs/import-probe.log\').write_text(probe.stdout, encoding=\'utf-8\')\n    if probe.returncode:\n        raise RuntimeError(probe.stdout[-3000:])\n    info = {\'python\': sys.version, \'torch\': torch.__version__, \'cuda\': torch.version.cuda,\n        \'scratch_free_gib\': free / 2**30, \'protected\': protected, \'pins\': packages,\n        \'system_packages_replaced\': False, \'gpu\': [\n            {\'index\': i, \'name\': torch.cuda.get_device_name(i),\n             \'capability\': list(torch.cuda.get_device_capability(i)),\n             \'vram_gib\': torch.cuda.get_device_properties(i).total_memory / 2**30}\n            for i in range(2)]}\n    atomic_json(run / \'environment.json\', info)\n    print(\'TWO T4:\', info[\'gpu\'], flush=True)\n    return str(python)\n', 'common.py': '"""Portable, non-destructive animation pilot contracts and media checks."""\nimport hashlib\nimport json\nimport os\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(2**20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    os.replace(temporary, path)\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef validate_config(config):\n    width, height = config[\'WIDTH\'], config[\'HEIGHT\']\n    if width % 32 or height % 32 or width * 9 != height * 16:\n        raise ValueError(\'Use exact 16:9 dimensions divisible by 32, e.g. 512x288 or 1024x576.\')\n    if config[\'FRAMES\'] < 9 or (config[\'FRAMES\'] - 1) % 4:\n        raise ValueError(\'Wan frame count must be 4n+1.\')\n    if config[\'FPS\'] != 24 or len(config[\'SEEDS\']) != 2:\n        raise ValueError(\'This controlled pilot uses 24 fps and exactly two seeds.\')\n    if not 1 <= config[\'MAX_GPU_WORKERS\'] <= 2:\n        raise ValueError(\'At most two independent GPU workers are allowed.\')\n    if not 10 <= config[\'STEPS\'] <= 80:\n        raise ValueError(\'Use 10..80 denoising steps; the pilot default is 40.\')\n\n\ndef encode_frames(frames, target, fps=24, ffmpeg=\'ffmpeg\'):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    frames = iter(frames)\n    first = np.asarray(next(frames), dtype=np.uint8)\n    height, width = first.shape[:2]\n    temporary = target.with_name(target.stem + \'.partial.mp4\')\n    command = [str(ffmpeg), \'-hide_banner\', \'-loglevel\', \'error\', \'-y\',\n               \'-f\', \'rawvideo\', \'-pix_fmt\', \'rgb24\', \'-s\', f\'{width}x{height}\', \'-r\', str(fps),\n               \'-i\', \'pipe:0\', \'-an\', \'-vf\', \'scale=in_range=full:out_range=tv:out_color_matrix=bt709\',\n               \'-c:v\', \'libx264\', \'-preset\', \'medium\', \'-crf\', \'16\',\n               \'-pix_fmt\', \'yuv420p\', \'-color_primaries\', \'bt709\', \'-color_trc\', \'bt709\',\n               \'-colorspace\', \'bt709\', \'-movflags\', \'+faststart\', str(temporary)]\n    error_path = target.with_suffix(\'.encoder.log\')\n    with error_path.open(\'wb\') as errors:\n        process = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=errors)\n        try:\n            process.stdin.write(first.tobytes())\n            for frame in frames:\n                array = np.asarray(frame, dtype=np.uint8)\n                if array.shape != first.shape:\n                    raise ValueError(\'Frame dimensions changed.\')\n                process.stdin.write(array.tobytes())\n            process.stdin.close()\n            if process.wait(timeout=300):\n                raise RuntimeError(error_path.read_text(encoding=\'utf-8\', errors=\'replace\')[-1500:])\n        except BaseException:\n            if process.poll() is None:\n                process.kill()\n            process.wait()\n            raise\n    os.replace(temporary, target)\n\n\ndef bilinear(image, x, y):\n    """Backward sampling: no synthesized disocclusions or hidden inpainting."""\n    height, width = image.shape[:2]\n    x = np.clip(x, 0, width - 1.001)\n    y = np.clip(y, 0, height - 1.001)\n    left, top = x.astype(np.int32), y.astype(np.int32)\n    fx, fy = (x - left)[..., None], (y - top)[..., None]\n    a = image[top, left].astype(np.float32)\n    b = image[top, left + 1].astype(np.float32)\n    c = image[top + 1, left].astype(np.float32)\n    d = image[top + 1, left + 1].astype(np.float32)\n    return np.clip((a * (1 - fx) + b * fx) * (1 - fy) +\n                   (c * (1 - fx) + d * fx) * fy, 0, 255).round().astype(np.uint8)\n\n\ndef control_frames(image, frames, depth=None):\n    """Static-camera and modest depth-guided warp controls, not true layered 3D."""\n    image = np.asarray(image.convert(\'RGB\'))\n    height, width = image.shape[:2]\n    y, x = np.mgrid[:height, :width].astype(np.float32)\n    if depth is None:\n        for _ in range(frames):\n            yield image\n        return\n    # A 3% safety crop contains the translation. Warp amplitude stays below 2 px at 512w.\n    for index in range(frames):\n        t = index / max(1, frames - 1)\n        progress = t * t * (3 - 2 * t)\n        shift = (progress - .5) * width * .006 * (.2 + .8 * depth)\n        sample_x = (x - width / 2) / 1.03 + width / 2 - shift\n        sample_y = (y - height / 2) / 1.03 + height / 2\n        yield bilinear(image, sample_x, sample_y)\n\n\ndef inspect_video(path, config, reference=None, ffmpeg=\'ffmpeg\', ffprobe=\'ffprobe\'):\n    raw = subprocess.check_output([str(ffprobe), \'-v\', \'error\', \'-count_frames\',\n        \'-show_streams\', \'-show_format\', \'-of\', \'json\', str(path)], text=True)\n    info = json.loads(raw)\n    videos = [s for s in info[\'streams\'] if s[\'codec_type\'] == \'video\']\n    if len(videos) != 1 or any(s[\'codec_type\'] == \'audio\' for s in info[\'streams\']):\n        raise ValueError(\'Asset must contain one silent video stream.\')\n    video = videos[0]\n    if (video[\'width\'], video[\'height\']) != (config[\'WIDTH\'], config[\'HEIGHT\']):\n        raise ValueError(\'Wrong native video resolution.\')\n    if video[\'avg_frame_rate\'] != \'24/1\' or int(video[\'nb_read_frames\']) != config[\'FRAMES\']:\n        raise ValueError(\'Wrong frame rate or count.\')\n    if abs(float(info[\'format\'][\'duration\']) - config[\'FRAMES\'] / config[\'FPS\']) > .06:\n        raise ValueError(\'Wrong clip duration.\')\n    process = subprocess.Popen([str(ffmpeg), \'-hide_banner\', \'-loglevel\', \'error\', \'-i\', str(path),\n        \'-vf\', \'scale=256:144\', \'-pix_fmt\', \'rgb24\', \'-f\', \'rawvideo\', \'pipe:1\'],\n        stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    size = 256 * 144 * 3\n    means, differences, reference_differences, sampled = [], [], [], []\n    previous = None\n    base = None if reference is None else np.asarray(reference.resize((256, 144)), dtype=np.float32)\n    while True:\n        data = process.stdout.read(size)\n        if not data:\n            break\n        if len(data) != size:\n            raise ValueError(\'Truncated decoded frame.\')\n        frame = np.frombuffer(data, np.uint8).reshape(144, 256, 3).astype(np.float32)\n        means.append(float(frame.mean()))\n        if previous is not None:\n            differences.append(float(np.abs(frame - previous).mean()))\n        if base is not None:\n            reference_differences.append(float(np.abs(frame - base).mean()))\n        if len(means) - 1 in {0, config[\'FRAMES\']//4, config[\'FRAMES\']//2,\n                              3*config[\'FRAMES\']//4, config[\'FRAMES\']-1}:\n            sampled.append(frame.astype(np.uint8))\n        previous = frame\n    errors = process.stderr.read().decode(\'utf-8\', errors=\'replace\')\n    process.stdout.close()\n    process.stderr.close()\n    if process.wait(timeout=60) or errors.strip() or len(means) != config[\'FRAMES\']:\n        raise ValueError(\'Full decode failed: \' + errors[-1000:])\n    report = {\'sha256\': sha256(path), \'decoded_frames\': len(means),\n        \'native_size\': [video[\'width\'], video[\'height\']], \'fps\': 24,\n        \'duration_seconds\': float(info[\'format\'][\'duration\']), \'silent\': True,\n        \'mean_rgb_range\': [min(means), max(means)],\n        \'frame_difference_mean\': float(np.mean(differences)),\n        \'frame_difference_max\': max(differences),\n        \'reference_rgb_difference_range\': None if base is None else\n            [min(reference_differences), max(reference_differences)],\n        \'technical_pass\': True, \'editorial_approved\': False,\n        \'warning\': \'Pixel differences are advisory, not identity or animation-quality verification.\'}\n    if min(means) < .4:\n        report[\'warning_black_frames\'] = True\n    return report, sampled\n'}
# Stage contract code in scratch, then validate the attached LOCAL asset bundle.
STAGING = SCRATCH/'contract'
STAGING.mkdir(parents=True,exist_ok=True)
for name,source in BUNDLE.items():
    (STAGING/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(STAGING))
from contract import identity,load_inputs,save_json
INPUTS = load_inputs(Path('/kaggle/input'),EXPECTED_HANDOFF_ID,SCRATCH/'inputs'/EXPECTED_HANDOFF_ID)
if len({r['config']['MAX_SEQUENCE'] for r in INPUTS['requests']}) != 1:
    raise ValueError('All requests in this batch must use the same text context length.')
signature = identity({'inputs':INPUTS,'scripts':BUNDLE})
RUN = Path('/kaggle/working')/('video-only-'+signature[:16])
for folder in ('scripts','frames','source','embeddings','clips','reports','requests','diagnostics','logs'):
    (RUN/folder).mkdir(parents=True,exist_ok=True)
for name,source in BUNDLE.items():
    (RUN/'scripts'/name).write_text(source,encoding='utf-8')
# Copy only checksum-bound frames. No TTS, ASR, music, titles or montage here.
load_inputs(SCRATCH/'inputs',EXPECTED_HANDOFF_ID,RUN)
JOB = {'run':str(RUN),'fingerprint':signature,'inputs':INPUTS,
    'models':INPUTS['profile']['models'],'config':INPUTS['requests'][0]['config'],
    'model_cache':str(SCRATCH/'models'),'negative_prompt':INPUTS['negative_prompt']}
save_json(RUN/'job.json',JOB)
from bootstrap import bootstrap
PYTHON = bootstrap(RUN,SCRATCH,INPUTS['profile']['pins'])
print('Kaggle stage: silent video only | batch:',EXPECTED_HANDOFF_ID)


In [ ]:
from driver import execute
RESULTS = execute(JOB,PYTHON)

In [ ]:
from IPython.display import FileLink,Video,display
print(json.dumps(json.loads((RUN/"video-results.json").read_text()),indent=2))
display(FileLink(str(RESULTS)))
for clip in sorted((RUN/"clips").glob("*.mp4")):
    display(Video(str(clip),embed=False,width=768))